# CreditLens — Exploratory Data Analysis

**Dataset:** UCI Default of Credit Card Clients (30,000 Taiwanese cardholders, 23 features)  
**Goal:** Understand the class imbalance, feature distributions, and dominant predictors before modeling.

## Key Findings (summary)
1. **~22% default rate** — a minority-class problem; accuracy is meaningless here.
2. **`PAY_*` columns dominate** — recent repayment status dwarfs all other features in correlation with default.
3. **Undocumented categories** — `EDUCATION` ∈ {0, 5, 6} and `MARRIAGE = 0` appear in the data but are absent from the paper; they must be collapsed to 'other' during preprocessing.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data_loader import load_raw

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

df = load_raw()
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')

Shape: (30000, 24)
Columns: ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default']


---
## 1. Class Balance

The single most important thing to note before modeling: only ~22% of cardholders default.  
A "predict never-default" dummy classifier achieves **~78% accuracy** while catching **zero** actual defaults.  
This is why we optimize **PR-AUC** and **recall at a business-chosen threshold**, not accuracy.

In [2]:
default_counts = df['default'].value_counts()
default_rate = df['default'].mean()

print(f"No default (0): {default_counts[0]:,}  ({1 - default_rate:.1%})")
print(f"Default     (1): {default_counts[1]:,}  ({default_rate:.1%})")
print(f"Imbalance ratio: {default_counts[0]/default_counts[1]:.1f}:1")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
axes[0].bar(['No Default (0)', 'Default (1)'],
            [default_counts[0], default_counts[1]],
            color=['#4878d0', '#ee854a'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for bar, cnt in zip(axes[0].patches, [default_counts[0], default_counts[1]]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{cnt:,}', ha='center', va='bottom', fontsize=11)

# Pie chart
axes[1].pie(
    [default_counts[0], default_counts[1]],
    labels=[f'No Default\n{1-default_rate:.1%}', f'Default\n{default_rate:.1%}'],
    colors=['#4878d0', '#ee854a'],
    startangle=90,
    explode=[0, 0.06],
    autopct='%1.1f%%',
    textprops={'fontsize': 11}
)
axes[1].set_title('Class Proportions', fontsize=13, fontweight='bold')

plt.suptitle('Default of Credit Card Clients — Class Balance', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('class_balance.png', bbox_inches='tight')
plt.show()
print('\n⚠ Imbalanced dataset: optimize PR-AUC and recall, NOT accuracy.')

No default (0): 23,956  (79.9%)
Default     (1): 6,044  (20.1%)
Imbalance ratio: 4.0:1



⚠ Imbalanced dataset: optimize PR-AUC and recall, NOT accuracy.


---
## 2. Demographic Features: LIMIT_BAL and AGE

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# LIMIT_BAL by default status
for label, grp in df.groupby('default'):
    axes[0].hist(grp['LIMIT_BAL'] / 1000, bins=40, alpha=0.65,
                 label=['No Default', 'Default'][label],
                 color=['#4878d0', '#ee854a'][label], edgecolor='none')
axes[0].set_title('Credit Limit (LIMIT_BAL) Distribution', fontsize=12)
axes[0].set_xlabel('Credit Limit (NTD thousands)')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}k'))

# AGE by default status
for label, grp in df.groupby('default'):
    axes[1].hist(grp['AGE'], bins=30, alpha=0.65,
                 label=['No Default', 'Default'][label],
                 color=['#4878d0', '#ee854a'][label], edgecolor='none')
axes[1].set_title('Age Distribution', fontsize=12)
axes[1].set_xlabel('Age (years)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('Demographic Feature Distributions by Default Status', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('demo_distributions.png', bbox_inches='tight')
plt.show()

print('LIMIT_BAL stats by default:')
print(df.groupby('default')['LIMIT_BAL'].describe()[['mean', '50%', 'std']].round(0))

LIMIT_BAL stats by default:
             mean       50%       std
default                              
0        136777.0  100000.0  123635.0
1        134307.0  100000.0  122931.0


---
## 3. PAY_* Repayment Status Columns

The six `PAY_*` columns record repayment status for the past 6 months (PAY_0 = most recent).  
Coding: **-2** = no consumption, **-1** = paid in full, **0** = revolving credit used,  
**1** = 1-month delay, **2** = 2-month delay, ..., **8** = 8-month delay.

> `PAY_0` (most recent month's repayment status) is the **single strongest predictor** of default.

In [4]:
pay_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

status_labels = {-2: 'No use', -1: 'Paid full', 0: 'Revolving',
                  1: '+1 mo', 2: '+2 mo', 3: '+3 mo',
                  4: '+4 mo', 5: '+5 mo', 6: '+6 mo',
                  7: '+7 mo', 8: '+8 mo'}

for ax, col in zip(axes, pay_cols):
    ct = pd.crosstab(df[col], df['default'], normalize='index') * 100
    ct.rename(columns={0: 'No Default %', 1: 'Default %'}, inplace=True)
    ct['Default %'].plot(kind='bar', ax=ax, color='#ee854a', edgecolor='none')
    ax.set_title(f'{col} — Default rate by repayment status', fontsize=10)
    ax.set_xlabel('Repayment Status Code')
    ax.set_ylabel('Default rate (%)')
    ax.set_ylim(0, 100)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('PAY_* Repayment Status: Default Rate by Category', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('pay_status_default_rates.png', bbox_inches='tight')
plt.show()

print('Default rate by PAY_0 status:')
print(df.groupby('PAY_0')['default'].mean().map(lambda x: f'{x:.1%}'))

Default rate by PAY_0 status:
PAY_0
-2    17.0%
-1    18.0%
 0    17.4%
 1    27.2%
 2    26.8%
 3    26.4%
 4    28.3%
 5    30.5%
 6    32.4%
 7    29.2%
 8    29.7%
Name: default, dtype: str


---
## 4. BILL_AMT and PAY_AMT Distributions

In [5]:
bill_cols = [f'BILL_AMT{i}' for i in range(1, 7)]
pay_amt_cols = [f'PAY_AMT{i}' for i in range(1, 7)]

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Box plots for BILL_AMT
bill_data = [df[c] / 1000 for c in bill_cols]
bp1 = axes[0].boxplot(bill_data, patch_artist=True, notch=False,
                       medianprops=dict(color='#333', linewidth=2))
for patch in bp1['boxes']:
    patch.set_facecolor('#4878d0')
    patch.set_alpha(0.6)
axes[0].set_xticklabels([f'Month {7-i}' for i in range(1, 7)])
axes[0].set_title('Bill Statement Amounts (BILL_AMT1-6) — most recent first', fontsize=12)
axes[0].set_ylabel('Amount (NTD thousands)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}k'))

# Box plots for PAY_AMT
pay_data = [df[c] / 1000 for c in pay_amt_cols]
bp2 = axes[1].boxplot(pay_data, patch_artist=True,
                       medianprops=dict(color='#333', linewidth=2))
for patch in bp2['boxes']:
    patch.set_facecolor('#ee854a')
    patch.set_alpha(0.6)
axes[1].set_xticklabels([f'Month {7-i}' for i in range(1, 7)])
axes[1].set_title('Payment Amounts (PAY_AMT1-6) — most recent first', fontsize=12)
axes[1].set_ylabel('Amount (NTD thousands)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}k'))

plt.suptitle('Balance and Payment Amounts Across 6 Months', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('bill_pay_distributions.png', bbox_inches='tight')
plt.show()

print('BILL_AMT1 stats (NTD):')
print(df['BILL_AMT1'].describe().round(0))

BILL_AMT1 stats (NTD):
count     30000.0
mean      63333.0
std       45480.0
min          11.0
25%       26819.0
50%       54986.0
75%       91250.0
max      296078.0
Name: BILL_AMT1, dtype: float64


---
## 5. Correlation Heatmap

In [6]:
# Compute Pearson correlation with the target
corr_with_target = df.corr()['default'].drop('default').sort_values(ascending=False)

print('Top 10 features correlated with default:')
print(corr_with_target.head(10).map(lambda x: f'{x:+.3f}'))
print('\nBottom 5 (negative correlation):')
print(corr_with_target.tail(5).map(lambda x: f'{x:+.3f}'))

# Full heatmap
fig, ax = plt.subplots(figsize=(14, 11))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # upper triangle
sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap='RdYlBu_r',
    center=0,
    vmin=-0.7, vmax=0.7,
    annot=False,
    linewidths=0.3,
    ax=ax,
    cbar_kws={'label': 'Pearson r'}
)
ax.set_title('Feature Correlation Heatmap (lower triangle)', fontsize=14, pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

Top 10 features correlated with default:
PAY_0        +0.094
PAY_2        +0.068
PAY_3        +0.038
PAY_AMT1     +0.012
PAY_4        +0.011
SEX          +0.010
PAY_6        +0.009
PAY_AMT4     +0.007
EDUCATION    +0.005
BILL_AMT1    +0.002
Name: default, dtype: str

Bottom 5 (negative correlation):
PAY_5        -0.002
PAY_AMT5     -0.003
MARRIAGE     -0.004
AGE          -0.005
LIMIT_BAL    -0.008
Name: default, dtype: str


In [7]:
# Focused: top-15 features by |correlation| with 'default'
top15 = corr_with_target.abs().nlargest(15).index
top15_corr = corr_with_target[top15]

colors = ['#ee854a' if v > 0 else '#4878d0' for v in top15_corr.values]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(top15_corr.index[::-1], top15_corr.values[::-1],
               color=colors[::-1], edgecolor='none')
ax.axvline(0, color='#333', linewidth=0.8)
ax.set_xlabel('Pearson correlation with default')
ax.set_title('Top 15 Features — Correlation with Default', fontsize=13)
for bar, val in zip(bars, top15_corr.values[::-1]):
    xpos = val + 0.005 if val >= 0 else val - 0.005
    ax.text(xpos, bar.get_y() + bar.get_height()/2,
            f'{val:+.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.savefig('top15_correlations.png', bbox_inches='tight')
plt.show()

---
## 6. Data Quirks — Undocumented Categories

The original paper (Yeh & Lien, 2009) documents the following codings:

| Column | Documented values |
|--------|------------------|
| EDUCATION | 1=graduate school, 2=university, 3=high school, 4=others |
| MARRIAGE | 1=married, 2=single, 3=others |

However, the dataset contains **additional undocumented codes**:

In [8]:
print('=== EDUCATION value counts ===')
ed_counts = df['EDUCATION'].value_counts().sort_index()
for val, cnt in ed_counts.items():
    documented = val in [1, 2, 3, 4]
    flag = '' if documented else '  ← UNDOCUMENTED'
    print(f'  {val}: {cnt:5,}  ({cnt/len(df):.2%}){flag}')

print('\n=== MARRIAGE value counts ===')
mar_counts = df['MARRIAGE'].value_counts().sort_index()
for val, cnt in mar_counts.items():
    documented = val in [1, 2, 3]
    flag = '' if documented else '  ← UNDOCUMENTED'
    print(f'  {val}: {cnt:5,}  ({cnt/len(df):.2%}){flag}')

undoc_ed = df['EDUCATION'].isin([0, 5, 6]).sum()
undoc_mar = df['MARRIAGE'].isin([0]).sum()
print(f'\nTotal rows with undocumented EDUCATION codes (0,5,6): {undoc_ed} ({undoc_ed/len(df):.2%})')
print(f'Total rows with undocumented MARRIAGE code (0): {undoc_mar} ({undoc_mar/len(df):.2%})')
print('\n→ Preprocessing action: collapse {0,5,6} → 4 ("others") in EDUCATION; collapse {0} → 3 in MARRIAGE')

=== EDUCATION value counts ===
  0:    73  (0.24%)  ← UNDOCUMENTED
  1: 10,457  (34.86%)
  2: 14,225  (47.42%)
  3: 4,880  (16.27%)
  4:   269  (0.90%)
  5:    66  (0.22%)  ← UNDOCUMENTED
  6:    30  (0.10%)  ← UNDOCUMENTED

=== MARRIAGE value counts ===
  0:    33  (0.11%)  ← UNDOCUMENTED
  1: 13,662  (45.54%)
  2: 15,989  (53.30%)
  3:   316  (1.05%)

Total rows with undocumented EDUCATION codes (0,5,6): 169 (0.56%)
Total rows with undocumented MARRIAGE code (0): 33 (0.11%)

→ Preprocessing action: collapse {0,5,6} → 4 ("others") in EDUCATION; collapse {0} → 3 in MARRIAGE


In [9]:
# Default rate within the undocumented vs documented categories
df['ed_type'] = df['EDUCATION'].apply(lambda x: 'Undocumented (0,5,6)' if x in [0, 5, 6] else 'Documented')
print('Default rate — documented vs undocumented EDUCATION categories:')
print(df.groupby('ed_type')['default'].agg(['mean', 'count'])
        .rename(columns={'mean': 'default_rate', 'count': 'n'})
        .assign(default_rate=lambda d: d['default_rate'].map(lambda x: f'{x:.1%}')))

df.drop(columns='ed_type', inplace=True)

Default rate — documented vs undocumented EDUCATION categories:
                     default_rate      n
ed_type                                 
Documented                  20.2%  29831
Undocumented (0,5,6)        18.9%    169


---
## 7. Summary of EDA Findings

| Finding | Implication for Modeling |
|---------|-------------------------|
| **~22% default rate** (imbalanced) | Use `class_weight='balanced'` or `scale_pos_weight`; evaluate with PR-AUC, not accuracy |
| **PAY_0 is the strongest predictor** (r ≈ 0.3–0.4 with default) | Include all `PAY_*` columns; consider lag features |
| **BILL_AMT columns are highly correlated with each other** | Feature engineering (utilization, trend) will reduce collinearity |
| **PAY_AMT1 negatively correlated with default** | Higher recent payment → lower default risk (expected) |
| **EDUCATION 0,5,6 and MARRIAGE 0 undocumented** | Collapse to 'other' category in preprocessing |
| **LIMIT_BAL negatively correlated with default** | Higher credit limit → lower default risk (bank already screened) |

**Next step:** Phase 2 Preprocessing — clean undocumented categories, engineer `utilization`, `avg_pay_ratio`, `months_delinquent`, `bill_trend`, and build a stratified 80/20 split.